# Part 3 — TabPFN, OOD (full), label noise, conformal, post-hoc analyses
**Calibration vs Deep Ensembles, MAKE revision** — *patched build v2.1*

## 1. Environment setup

In [ ]:
# Working directory for cached probs and intermediate CSVs.
# Override with:  export WORK_DIR=/path/to/persistent/storage
import os, pathlib

WORK_DIR = pathlib.Path(os.environ.get('WORK_DIR', './work')).resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)
PROBS_DIR = WORK_DIR / 'probs'
PROBS_DIR.mkdir(exist_ok=True)

print(f"Working directory: {WORK_DIR}")
print(f"Cached probability files: {len(list(PROBS_DIR.glob('*.npz')))}")


In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32       = True
    torch.backends.cudnn.benchmark        = True
    print("  TF32 + cudnn.benchmark enabled.")

In [ ]:
# Install dependencies 
!pip install catboost dirichletcal statsmodels
!pip install openml==0.15.1 lightgbm==4.3.0 'xgboost>=2.0.0' 'scikit-learn>=1.3.0'

# TabPFN install — may take a moment due to large prior weights
TABPFN_AVAILABLE = False
try:
    !pip install tabpfn
    import tabpfn
    print(f"✓ tabpfn {tabpfn.__version__}")
    TABPFN_AVAILABLE = True
except Exception as exc:
    print(f"⚠ tabpfn install failed: {exc}")
    print("  Part 3 will skip the TabPFN section but proceed with other analyses.")

import catboost, dirichletcal
print(f"✓ catboost {catboost.__version__}, dirichletcal {dirichletcal.__version__}")

## 2. Get codebase + load existing artifacts

In [ ]:
# Locate the repo root (the directory containing src/) and add it to sys.path.
# Works regardless of whether the notebook is run from notebooks/, the repo root,
# or one level deeper.
import os, pathlib, sys

def _find_repo_root(start=None):
    p = pathlib.Path(start or os.getcwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / 'src').is_dir():
            return cand
    raise FileNotFoundError(
        "Could not find a 'src/' directory in the current working directory or any "
        "parent. Run this notebook from inside the cloned repository."
    )

REPO_ROOT = _find_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f"Repo root: {REPO_ROOT}")
print("✓ src/ found")

from src.datasets import load_task, split_dataset
from src.calibration import build_calibrator, CALIBRATOR_LABELS, BaseCalibrator, TemperatureScaling
from src.metrics import evaluate_all
from src.models import LightGBMModel, XGBoostModel, SingleMLP, DeepEnsemble, get_device
from src.utils import EPS
print("✓ src/ imports OK")


In [ ]:
import pandas as pd
import numpy as np

manifest = pd.read_csv(WORK_DIR / 'dataset_manifest.csv')
results  = pd.read_csv(WORK_DIR / 'results_raw.csv')

print(f"Manifest: {len(manifest)} datasets")
print(f"Existing results: {len(results)} rows")
print(f"  models: {sorted(results['model'].unique())}")
print(f"  calibrators: {sorted(results['calibrator'].unique())}")
device = get_device()

## 3. TabPFN v2 inference 

TabPFN v2 is a pretrained transformer for tabular classification via in-context learning. Constraints:
- ≤10,000 training samples
- ≤500 features  
- ≤10 classes

For datasets exceeding constraints we either skip (most defensible) or subsample training set (less rigorous). We **skip** incompatible datasets and report TabPFN results as a stratified analysis.

In [ ]:
# Define CatBoost + Dirichlet (needed for the run loop to be self-contained)
from catboost import CatBoostClassifier

class CatBoostModel:
    _DEFAULT_PARAMS = {"learning_rate":0.05,"depth":6,"min_data_in_leaf":20,
                       "rsm":0.8,"subsample":0.8,"bootstrap_type":"Bernoulli",
                       "l2_leaf_reg":3.0,"od_type":"Iter","od_wait":50,
                       "verbose":False,"thread_count":-1,"allow_writing_files":False}
    def __init__(self, n_classes, num_boost_round=500, extra_params=None):
        self.n_classes=n_classes; p=dict(self._DEFAULT_PARAMS); p["iterations"]=num_boost_round
        if n_classes>2: p.update({"loss_function":"MultiClass","eval_metric":"MultiClass","classes_count":n_classes})
        else: p.update({"loss_function":"Logloss","eval_metric":"Logloss"})
        if extra_params: p.update(extra_params)
        self._params=p; self._model=None
    def fit(self, X_train, y_train, X_val=None, y_val=None):
        self._model = CatBoostClassifier(**self._params)
        es = (X_val,y_val) if X_val is not None else None
        self._model.fit(X_train, y_train, eval_set=es, verbose=False, plot=False)
        return self
    def predict_proba(self, X):
        return np.clip(self._model.predict_proba(X), EPS, 1.0)

from dirichletcal.calib.fulldirichlet import FullDirichletCalibrator
class DirichletODIRCalibrator(BaseCalibrator):
    def __init__(self, reg_lambda=1e-3, reg_mu=1e-3):
        self.reg_lambda=reg_lambda; self.reg_mu=reg_mu; self._model=None
    def fit(self, probs_val, y_val):
        self._model = FullDirichletCalibrator(reg_lambda=self.reg_lambda, reg_mu=self.reg_mu)
        self._model.fit(probs_val, y_val); return self
    def calibrate(self, probs):
        return np.clip(self._model.predict_proba(probs), EPS, 1.0-EPS)

def build_calibrator_extended(name):
    if name == "dirichlet": return DirichletODIRCalibrator()
    return build_calibrator(name)
print("✓ CatBoost + Dirichlet redefined")

In [ ]:
# Identify TabPFN-compatible datasets
def tabpfn_compatible(task_meta):
    """TabPFN v2 constraints: ≤10k samples, ≤500 features, ≤10 classes.
    Use 54% training fraction to estimate train-set size."""
    n_train_est = task_meta["n_samples"] * 0.54
    return (n_train_est <= 10_000 and
            task_meta["n_features"] <= 500 and
            task_meta["n_classes"] <= 10)

compatible = manifest[manifest.apply(tabpfn_compatible, axis=1)]
incompatible = manifest[~manifest.apply(tabpfn_compatible, axis=1)]
print(f"TabPFN-compatible datasets: {len(compatible)} / {len(manifest)}")
print(f"Incompatible (will be skipped):")
print(incompatible[["name","n_samples","n_features","n_classes"]].to_string(index=False))
print(f"\nCompatible:")
print(compatible[["name","n_samples","n_features","n_classes"]].to_string(index=False))

In [ ]:
# TabPFN wrapper to match existing model interface
class TabPFNModel:
    """Wrapper around TabPFNClassifier matching the existing model API.
    Uses default settings of tabpfn v2."""
    def __init__(self, n_classes, device="cuda"):
        self.n_classes = n_classes
        self._device = device
        self._model = None
    def fit(self, X_train, y_train, X_val=None, y_val=None):
        from tabpfn import TabPFNClassifier
        # Auto-detect available device
        self._model = TabPFNClassifier(device=self._device, ignore_pretraining_limits=False)
        self._model.fit(X_train, y_train)
        return self
    def predict_proba(self, X):
        return np.clip(self._model.predict_proba(X), EPS, 1.0)

if TABPFN_AVAILABLE:
    print("✓ TabPFN wrapper defined")
else:
    print("⚠ Skipping TabPFN — install failed above")

In [ ]:
# TabPFN API token 
import os, getpass
if TABPFN_AVAILABLE and not os.environ.get("TABPFN_TOKEN"):
    os.environ["TABPFN_TOKEN"] = getpass.getpass("Paste TABPFN_TOKEN (input hidden): ").strip()
    print("✓ Token set for this session.")
elif os.environ.get("TABPFN_TOKEN"):
    print("✓ TABPFN_TOKEN already in environment.")
else:
    print("⚠ TabPFN unavailable — skipping token prompt.")

In [ ]:
# Run TabPFN on compatible datasets, all 5 calibrators, 5 seeds
import time
TABPFN_SEEDS = [0, 1, 2, 3, 4]
TABPFN_CALIBRATORS = ["none", "temp", "logistic", "isotonic", "dirichlet"]

tabpfn_results = []
tabpfn_failures = []

if TABPFN_AVAILABLE and len(compatible) > 0:
    compat_tasks = compatible["task_id"].tolist()
    print(f"Running TabPFN on {len(compat_tasks)} datasets × {len(TABPFN_SEEDS)} seeds")
    t0 = time.time()
    for i, task_id in enumerate(compat_tasks):
        data = load_task(task_id)
        if data is None:
            tabpfn_failures.append({"task_id":task_id, "reason":"load failed"})
            continue
        print(f"\n[{i+1}/{len(compat_tasks)}] {data['name']} n={data['n_samples']}")
        for seed in TABPFN_SEEDS:
            split = split_dataset(data, seed=seed)
            try:
                tt0 = time.time()
                m = TabPFNModel(n_classes=split["n_classes"], device="cuda" if torch.cuda.is_available() else "cpu")
                m.fit(split["X_train"], split["y_train"])
                train_time = time.time() - tt0
                probs_vc = m.predict_proba(split["X_val_cal"])
                probs_te = m.predict_proba(split["X_test"])
            except Exception as exc:
                print(f"    ✗ seed={seed} TabPFN failed: {exc}")
                tabpfn_failures.append({"task_id":task_id, "seed":seed, "reason":str(exc)})
                continue
            for cal_name in TABPFN_CALIBRATORS:
                try:
                    cal = build_calibrator_extended(cal_name)
                    cal.fit(probs_vc, split["y_val_cal"])
                    probs_cal = cal.calibrate(probs_te)
                    metrics     = evaluate_all(probs_cal, split["y_test"])
                    raw_metrics = evaluate_all(probs_te,  split["y_test"])
                    row = {
                        "task_id": task_id, "dataset_name": split["name"],
                        "n_samples": split["n_samples"], "n_train": split["n_train"],
                        "n_val_es": split["n_val_es"], "n_val_cal": split["n_val_cal"],
                        "n_test": split["n_test"], "n_features": split["n_features"],
                        "n_classes": split["n_classes"], "seed": seed,
                        "model": "tabpfn", "calibrator": cal_name,
                        "calibrator_label": (CALIBRATOR_LABELS.get(cal_name) if cal_name in CALIBRATOR_LABELS else "Dirichlet ODIR"),
                        **{f"cal_{k}":v for k,v in metrics.items()},
                        **{f"raw_{k}":v for k,v in raw_metrics.items()},
                        "train_time_s": round(train_time, 2),
                    }
                    tabpfn_results.append(row)
                except Exception as exc:
                    print(f"    ✗ seed={seed} {cal_name} cal failed: {exc}")
                    tabpfn_failures.append({"task_id":task_id, "seed":seed, "calibrator":cal_name, "reason":str(exc)})
        print(f"  ✓ done. {len(tabpfn_results)} rows so far, {(time.time()-t0)/60:.1f} min elapsed")
    print(f"\n══════════════════════════════════════")
    print(f"TabPFN: {len(tabpfn_results)} rows, {len(tabpfn_failures)} failures")
    print(f"Total: {(time.time()-t0)/60:.1f} min")
else:
    print("TabPFN section skipped.")

if tabpfn_results:
    pd.DataFrame(tabpfn_results).to_csv(WORK_DIR / "tabpfn_results.csv", index=False)
    print(f"✓ Saved: tabpfn_results.csv")

## 4. OOD evaluation — full coverage

Gaussian feature-noise injection on Xtest with σ ∈ {0.5, 1.0, 2.0} (in standardized-feature units; features have unit variance after `StandardScaler`).

**Scope:** All 36 datasets, seed=0 only, **all 8 models** (lgbm, xgboost, catboost, single_mlp, mc_dropout, deep_ensemble_m3, deep_ensemble, deep_ensemble_m10), 5 calibrators. Single-seed choice is justified by the 5-seed H1 stability already established on clean data; OOD variance comes mostly from the noise realization itself.

Total: 36 × 1 × 8 × 5 × 3 σ = **4,320 OOD rows**.


In [ ]:
# Full OOD: all 36 datasets
OOD_TASKS = manifest["task_id"].tolist()
print(f"OOD subset: ALL {len(OOD_TASKS)} datasets")

In [ ]:
# Define MC-Dropout class for OOD retraining (matches Part 2 definition)
import torch.nn as nn

class MCDropoutMLP(SingleMLP):
    def __init__(self, T=30, **kwargs):
        super().__init__(**kwargs); self.T = T
    def predict_proba(self, X):
        self._model.eval()
        for mod in self._model.modules():
            if isinstance(mod, nn.Dropout):
                mod.train()
        with torch.no_grad():
            Xt = torch.FloatTensor(X).to(self.device)
            samples = []
            for _ in range(self.T):
                samples.append(torch.softmax(self._model(Xt), dim=-1).cpu().numpy())
        return np.mean(np.stack(samples, axis=0), axis=0)

ENSEMBLE_SIZES = {"deep_ensemble": 5, "deep_ensemble_m3": 3, "deep_ensemble_m10": 10}

def build_ood_model(model_name, n_classes, n_features, seed):
    if model_name == "lgbm":     return LightGBMModel(n_classes=n_classes), {}
    if model_name == "xgboost":  return XGBoostModel(n_classes=n_classes), {}
    if model_name == "catboost": return CatBoostModel(n_classes=n_classes), {}
    if model_name == "single_mlp":
        return SingleMLP(input_dim=n_features, n_classes=n_classes, hidden_dims=(256,128),
                         lr=1e-3, epochs=200, batch_size=256, patience=20, device=device), {"seed":seed}
    if model_name == "mc_dropout":
        return MCDropoutMLP(T=30, input_dim=n_features, n_classes=n_classes, hidden_dims=(256,128),
                            lr=1e-3, epochs=200, batch_size=256, patience=20, device=device), {"seed":seed}
    if model_name in ENSEMBLE_SIZES:
        return DeepEnsemble(n_members=ENSEMBLE_SIZES[model_name],
                            input_dim=n_features, n_classes=n_classes, hidden_dims=(256,128),
                            lr=1e-3, epochs=200, batch_size=256, patience=20, device=device), {"base_seed":seed}
    raise ValueError(f"Unknown model: {model_name}")

print("✓ OOD model factory + MC-Dropout defined")

### 4a. Pre-rerun cleanup (one-time)

Run this **once** before the patched OOD execution. It:

1. Archives any existing `ood_results.csv` from the pre-fix run, so the contaminated numbers are preserved as `ood_results_pre_rng_fix.csv` for the response letter / paper trail.
2. Deletes any stale `part3_ood_partial.csv` so the rerun starts from a clean state and does not re-introduce the resume-induced noise inconsistency.
3. Creates the `ood_noise/` directory for saved noise tensors.


In [ ]:
# Pre-rerun cleanup — archive pre-fix artifacts and create noise dir
from datetime import datetime

# (a) Archive pre-fix ood_results.csv if present
old = WORK_DIR / "ood_results.csv"
if old.exists():
    archive = WORK_DIR / "ood_results_pre_rng_fix.csv"
    if not archive.exists():
        old.rename(archive)
        print(f"✓ Archived pre-fix results → {archive.name}")
    else:
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        alt = WORK_DIR / f"ood_results_pre_rng_fix_{ts}.csv"
        old.rename(alt)
        print(f"⚠ Existing archive present; saved current as {alt.name}")
else:
    print("✓ No prior ood_results.csv — nothing to archive.")

# (b) Delete stale partial checkpoint
partial = WORK_DIR / "part3_ood_partial.csv"
if partial.exists():
    partial.unlink()
    print(f"✓ Deleted stale partial checkpoint: {partial.name}")
else:
    print("✓ No stale partial checkpoint — starting clean.")

# (c) Archive any pre-v2 noise tensors. v1 of the patched build used
#     per-task-only seeding (task_id alone); v2 uses (task_id, sigma, OOD_SEED).
#     Cached tensors from v1 are NOT byte-identical to v2 tensors, so we move
#     the old directory aside rather than load it.
noise_dir = WORK_DIR / "ood_noise"
if noise_dir.exists() and any(noise_dir.glob("task_*.npz")):
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    archive_dir = WORK_DIR / f"ood_noise_pre_v2_{ts}"
    noise_dir.rename(archive_dir)
    n_moved = len(list(archive_dir.glob("task_*.npz")))
    print(f"✓ Archived pre-v2 noise tensors → {archive_dir.name}")
    print(f"  ({n_moved} files moved)")

# (d) Create a fresh, empty noise directory for the v2 run
noise_dir = WORK_DIR / "ood_noise"
noise_dir.mkdir(exist_ok=True)
n_files = len(list(noise_dir.glob("*.npz")))
print(f"✓ Noise tensors will be saved under: {noise_dir}")
print(f"  (currently empty: {n_files} files)")


In [ ]:
# Run full OOD evaluation across all 36 datasets, all 8 models, 1 seed, 3 σ levels
OOD_MODELS = ["lgbm", "xgboost", "catboost", "single_mlp", "mc_dropout",
              "deep_ensemble_m3", "deep_ensemble", "deep_ensemble_m10"]
OOD_CALIBRATORS = ["none", "temp", "logistic", "isotonic", "dirichlet"]
OOD_SIGMAS = [0.5, 1.0, 2.0]
OOD_SEED = 0

NOISE_DIR = WORK_DIR / "ood_noise"
NOISE_DIR.mkdir(exist_ok=True)


def get_or_make_noises(task_id, n_test, n_feat, sigmas, base_seed):
    """Return {sigma: noise} for this task. Load from disk if present; else generate
    deterministically from (task_id, sigma, base_seed) and persist."""
    noise_path = NOISE_DIR / f"task_{int(task_id)}.npz"
    keys_needed = [f"sigma_{s}" for s in sigmas]

    if noise_path.exists():
        try:
            cached = np.load(noise_path)
            if all(k in cached.files for k in keys_needed):
                noises = {s: cached[f"sigma_{s}"].astype(np.float32) for s in sigmas}
                shapes_ok = all(noises[s].shape == (n_test, n_feat) for s in sigmas)
                if shapes_ok:
                    return noises, "loaded"
            # Fall through: cached file is incomplete or shape-mismatched. Regenerate.
        except Exception:
            pass  # corrupt cache → regenerate

    noises = {}
    for s in sigmas:
        # Mixing constants so the seed depends on task_id, sigma, AND base_seed.
        seed = (int(task_id) * 100003 + int(round(s * 1000)) * 1009 + int(base_seed)) % (2**32)
        rng = np.random.RandomState(seed)
        noises[s] = (rng.randn(n_test, n_feat) * s).astype(np.float32)

    np.savez_compressed(noise_path, **{f"sigma_{s}": noises[s] for s in sigmas})
    return noises, "generated"


# Resume from partial OOD checkpoint if present.
# Because noise is now deterministic per (task_id, sigma, OOD_SEED) and reloaded
# from disk when available, resume is safe.
ood_partial = WORK_DIR / "part3_ood_partial.csv"
if ood_partial.exists():
    prev = pd.read_csv(ood_partial)
    ood_results = prev.to_dict("records")
    ood_done = {(int(r["task_id"]), r["model"], r["calibrator"], float(r["ood_sigma"]))
                for r in ood_results}
    print(f"Resumed OOD: {len(ood_results)} rows already done.")
else:
    ood_results = []
    ood_done = set()

ood_failures = []
t0 = time.time()

for i, task_id in enumerate(OOD_TASKS):
    # Check if all cells for this task already done
    expected_cells = len(OOD_MODELS) * len(OOD_CALIBRATORS) * len(OOD_SIGMAS)
    done_this_task = sum(1 for k in ood_done if k[0] == task_id)
    if done_this_task >= expected_cells:
        continue

    data = load_task(int(task_id))
    if data is None:
        ood_failures.append({"task_id": int(task_id), "reason": "load failed"})
        continue
    print(f"\n[{i+1}/{len(OOD_TASKS)}] {data['name']} n={data['n_samples']}")
    split = split_dataset(data, seed=OOD_SEED)
    n_test = split["X_test"].shape[0]
    n_feat = split["X_test"].shape[1]

    # Deterministic noise per (task_id, sigma, OOD_SEED); loaded from disk if cached.
    noises, noise_origin = get_or_make_noises(task_id, n_test, n_feat, OOD_SIGMAS, OOD_SEED)
    print(f"    noise: {noise_origin}")

    for model_name in OOD_MODELS:
        # Skip if all (model, cal, sigma) for this task done
        model_done = sum(1 for k in ood_done if k[0] == task_id and k[1] == model_name)
        expected_per_model = len(OOD_CALIBRATORS) * len(OOD_SIGMAS)
        if model_done >= expected_per_model:
            continue
        try:
            # Peak GPU memory tracking per (task, model) for R1.9
            peak_mem_mb = None
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats()

            model, fit_kwargs = build_ood_model(model_name, split["n_classes"], split["n_features"], OOD_SEED)
            mt0 = time.time()
            model.fit(split["X_train"], split["y_train"], split["X_val_es"], split["y_val_es"], **fit_kwargs)
            train_time = time.time() - mt0

            probs_val_cal = model.predict_proba(split["X_val_cal"])
            calibrators = {}
            for cal_name in OOD_CALIBRATORS:
                cal = build_calibrator_extended(cal_name)
                cal.fit(probs_val_cal, split["y_val_cal"])
                calibrators[cal_name] = cal

            for sigma in OOD_SIGMAS:
                X_shifted = split["X_test"] + noises[sigma]
                probs_te_shifted = model.predict_proba(X_shifted)
                for cal_name, cal in calibrators.items():
                    key = (int(task_id), model_name, cal_name, sigma)
                    if key in ood_done:
                        continue
                    probs_cal = cal.calibrate(probs_te_shifted)
                    metrics     = evaluate_all(probs_cal,        split["y_test"])
                    raw_metrics = evaluate_all(probs_te_shifted, split["y_test"])
                    ood_results.append({
                        "task_id": int(task_id), "dataset_name": data["name"],
                        "n_samples": split["n_samples"], "n_features": split["n_features"],
                        "n_classes": split["n_classes"], "seed": OOD_SEED,
                        "model": model_name, "calibrator": cal_name, "ood_sigma": sigma,
                        **{f"cal_{k}": v for k, v in metrics.items()},
                        **{f"raw_{k}": v for k, v in raw_metrics.items()},
                        "train_time_s": round(train_time, 2),
                        "peak_gpu_mem_mb": peak_mem_mb,  # filled after inference, below
                    })
                    ood_done.add(key)

            # Capture peak memory once per (task, model), after all inferences for it
            if torch.cuda.is_available():
                peak_mem_mb = round(torch.cuda.max_memory_allocated() / (1024 * 1024), 2)
                # Backfill the rows just added for this (task, model)
                for row in ood_results[-(len(OOD_CALIBRATORS) * len(OOD_SIGMAS)):]:
                    if row["task_id"] == int(task_id) and row["model"] == model_name:
                        row["peak_gpu_mem_mb"] = peak_mem_mb

        except Exception as exc:
            print(f"    ✗ {model_name} failed: {exc}")
            ood_failures.append({"task_id": int(task_id), "model": model_name, "reason": str(exc)})

    # Checkpoint to WORK_DIR after every dataset
    pd.DataFrame(ood_results).to_csv(ood_partial, index=False)
    elapsed = time.time() - t0
    print(f"  ✓ {data['name']} done. {len(ood_results)} rows so far. Elapsed {elapsed/60:.1f}min")

print(f"\n══════════════════════════════════════")
print(f"OOD: {len(ood_results)} rows, {len(ood_failures)} failures")
expected_total = len(OOD_TASKS) * len(OOD_MODELS) * len(OOD_CALIBRATORS) * len(OOD_SIGMAS)
print(f"Expected total: {expected_total} rows")
if ood_results:
    pd.DataFrame(ood_results).to_csv(WORK_DIR / "ood_results.csv", index=False)
    if ood_partial.exists():
        ood_partial.unlink()
    print(f"✓ Saved: ood_results.csv (now includes peak_gpu_mem_mb column)")
if ood_failures:
    pd.DataFrame(ood_failures).to_csv(WORK_DIR / "part3_ood_failures.csv", index=False)
    print(f"⚠ Failures logged: part3_ood_failures.csv")


## 4b. Label-noise robustness 

Inject symmetric label noise into Xtrain at η ∈ {0.1, 0.2} — uniformly resample η fraction of training labels to random classes. Evaluate on representative subset (6 datasets, 1 seed, 8 models, 5 calibrators) to characterize calibration behavior under label corruption without doubling overall compute.

Total: 6 × 1 × 8 × 5 × 2 η = **480 noisy-label rows**.

In [ ]:
# Limited noisy-label scope: same 6 datasets as the original OOD diagnostic
NOISE_TASKS = []
for regime, n in [("small", 2), ("medium", 2), ("large", 2)]:
    cand = manifest[manifest.size_regime == regime].nsmallest(n, "n_samples")
    NOISE_TASKS.extend(cand["task_id"].tolist())
print(f"Noisy-label subset: {len(NOISE_TASKS)} datasets")

NOISE_LEVELS = [0.1, 0.2]
NOISE_MODELS = OOD_MODELS  # same 8 models
NOISE_CALIBRATORS = OOD_CALIBRATORS  # same 5 calibrators
NOISE_SEED = 0

noise_partial = WORK_DIR / "part3_noise_partial.csv"
if noise_partial.exists():
    prev = pd.read_csv(noise_partial)
    noise_results = prev.to_dict("records")
    noise_done = {(int(r["task_id"]), r["model"], r["calibrator"], float(r["label_noise_eta"]))
                  for r in noise_results}
    print(f"Resumed: {len(noise_results)} rows already done.")
else:
    noise_results = []
    noise_done = set()

noise_failures = []
t0 = time.time()

def inject_label_noise(y, eta, n_classes, rng):
    """Symmetric label noise: with prob eta, replace label with uniform random class."""
    y_noisy = y.copy()
    n = len(y)
    n_corrupt = int(eta * n)
    if n_corrupt == 0:
        return y_noisy
    corrupt_idx = rng.choice(n, size=n_corrupt, replace=False)
    new_labels = rng.randint(0, n_classes, size=n_corrupt)
    y_noisy[corrupt_idx] = new_labels
    return y_noisy

for i, task_id in enumerate(NOISE_TASKS):
    data = load_task(int(task_id))
    if data is None:
        noise_failures.append({"task_id":int(task_id), "reason":"load failed"})
        continue
    print(f"\n[{i+1}/{len(NOISE_TASKS)}] {data['name']} n={data['n_samples']}")
    split = split_dataset(data, seed=NOISE_SEED)

    for eta in NOISE_LEVELS:
        rng_noise = np.random.RandomState(int(task_id) + int(eta * 100))
        y_train_noisy = inject_label_noise(split["y_train"], eta, split["n_classes"], rng_noise)

        for model_name in NOISE_MODELS:
            # Skip if all (model, cal) at this eta done for this task
            done_count = sum(1 for k in noise_done if k[0]==int(task_id) and k[1]==model_name and abs(k[3]-eta)<1e-6)
            if done_count >= len(NOISE_CALIBRATORS):
                continue
            try:
                model, fit_kwargs = build_ood_model(model_name, split["n_classes"], split["n_features"], NOISE_SEED)
                mt0 = time.time()
                model.fit(split["X_train"], y_train_noisy, split["X_val_es"], split["y_val_es"], **fit_kwargs)
                train_time = time.time() - mt0

                # Calibrator fitted on clean Xval_cal predictions (realistic deployment)
                probs_val_cal = model.predict_proba(split["X_val_cal"])
                probs_test    = model.predict_proba(split["X_test"])

                for cal_name in NOISE_CALIBRATORS:
                    key = (int(task_id), model_name, cal_name, eta)
                    if key in noise_done:
                        continue
                    cal = build_calibrator_extended(cal_name)
                    cal.fit(probs_val_cal, split["y_val_cal"])
                    probs_cal = cal.calibrate(probs_test)
                    metrics     = evaluate_all(probs_cal,  split["y_test"])
                    raw_metrics = evaluate_all(probs_test, split["y_test"])
                    noise_results.append({
                        "task_id": int(task_id), "dataset_name": data["name"],
                        "n_samples": split["n_samples"], "n_features": split["n_features"],
                        "n_classes": split["n_classes"], "seed": NOISE_SEED,
                        "model": model_name, "calibrator": cal_name,
                        "label_noise_eta": eta,
                        **{f"cal_{k}":v for k,v in metrics.items()},
                        **{f"raw_{k}":v for k,v in raw_metrics.items()},
                        "train_time_s": round(train_time, 2),
                    })
                    noise_done.add(key)
            except Exception as exc:
                print(f"    ✗ {model_name} eta={eta} failed: {exc}")
                noise_failures.append({"task_id":int(task_id), "model":model_name,
                                       "eta":eta, "reason":str(exc)})

    pd.DataFrame(noise_results).to_csv(noise_partial, index=False)
    print(f"  ✓ {data['name']} done. {len(noise_results)} rows. {(time.time()-t0)/60:.1f}min")

print(f"\n══════════════════════════════════════")
print(f"Label noise: {len(noise_results)} rows, {len(noise_failures)} failures")
if noise_results:
    pd.DataFrame(noise_results).to_csv(WORK_DIR / "label_noise_results.csv", index=False)
    if noise_partial.exists():
        noise_partial.unlink()
    print(f"✓ Saved: label_noise_results.csv")

## 5. Conformal coverage analysis

Split conformal prediction on the saved probabilities. For each (dataset, model, seed):
1. Compute Adaptive Prediction Set (APS) conformity scores on Xval_cal using **raw** probabilities
2. Determine threshold from the (1−α) quantile, α=0.1 (target 90% coverage)
3. Form prediction sets on Xtest; report empirical coverage and average set size

In [ ]:
# Conformal helpers (cleaned: dead `aps_sets` stub removed in this build)
def aps_score(probs, y):
    """Adaptive Prediction Set (APS) conformity scores.
    For each (sample, true class), score = cumulative sorted prob mass
    up to and including the true class's probability."""
    n, K = probs.shape
    sorted_idx   = np.argsort(-probs, axis=1)
    sorted_probs = -np.sort(-probs, axis=1)
    cum_probs    = np.cumsum(sorted_probs, axis=1)
    scores = np.zeros(n)
    for i in range(n):
        rank = np.where(sorted_idx[i] == y[i])[0][0]
        scores[i] = cum_probs[i, rank]
    return scores

def conformal_eval(probs_val, y_val, probs_test, y_test, alpha=0.1):
    """Split conformal at coverage 1−alpha. Returns (empirical_coverage, avg_set_size)."""
    n_val = len(y_val)
    val_scores = aps_score(probs_val, y_val)
    q_level = np.ceil((n_val + 1) * (1 - alpha)) / n_val
    q_level = min(q_level, 1.0)
    threshold = np.quantile(val_scores, q_level, method="higher")

    n_test, K = probs_test.shape
    sorted_idx   = np.argsort(-probs_test, axis=1)
    sorted_probs = -np.sort(-probs_test, axis=1)
    cum_probs    = np.cumsum(sorted_probs, axis=1)

    covered, sizes = 0, 0
    for i in range(n_test):
        idx = np.searchsorted(cum_probs[i], threshold) + 1
        idx = min(idx, K)
        pred_set = set(sorted_idx[i, :idx].tolist())
        sizes += len(pred_set)
        if y_test[i] in pred_set:
            covered += 1
    return covered / n_test, sizes / n_test

print("✓ Conformal helpers defined")

In [ ]:
for p in dupes[:10]:
    name = pathlib.Path(p).name
    # Verify the non-duplicate version exists
    original = pathlib.Path(p).parent / re.sub(r"\s\(\d+\)\.npz$", ".npz", name)
    exists = "✓ original exists" if original.exists() else "✗ ORIGINAL MISSING"
    print(f"  {name}  →  {exists}")

In [ ]:
for p in dupes:
    pathlib.Path(p).unlink()
print(f"✓ Deleted {len(dupes)} duplicate files.")
print(f"Remaining: {len(list(PROBS_DIR.glob('*.npz')))} files")

In [ ]:
# Sweep all saved probs and compute conformal coverage
import glob

conformal_results = []
prob_files = sorted(glob.glob(str(PROBS_DIR / "*.npz")))
print(f"Found {len(prob_files)} cached prob files")

t0 = time.time()
for i, path in enumerate(prob_files):
    fname = pathlib.Path(path).stem
    # Skip member-probs files (we use averaged probs for conformal)
    if "__members" in fname:
        continue
    parts = fname.split("__")
    if len(parts) != 3:
        continue
    task_id, model_name, seed_str = parts
    seed = int(seed_str.replace("seed", ""))

    try:
        data = np.load(path)
        probs_vc = data["probs_val_cal"]
        probs_te = data["probs_test"]
        y_vc     = data["y_val_cal"]
        y_te     = data["y_test"]
        cov, sz  = conformal_eval(probs_vc, y_vc, probs_te, y_te, alpha=0.1)
        conformal_results.append({
            "task_id": int(task_id), "model": model_name, "seed": seed,
            "target_coverage": 0.9,
            "empirical_coverage": cov,
            "avg_set_size":      sz,
            "n_classes":         probs_te.shape[1],
            "n_test":            len(y_te),
        })
    except Exception as exc:
        print(f"  ✗ {fname}: {exc}")

    if (i+1) % 100 == 0:
        print(f"  {i+1}/{len(prob_files)} processed, {(time.time()-t0)/60:.1f} min")

print(f"\nConformal: {len(conformal_results)} (task, model, seed) cells")
if conformal_results:
    cf = pd.DataFrame(conformal_results)
    cf.to_csv(WORK_DIR / "conformal_results.csv", index=False)
    print(f"✓ Saved: conformal_results.csv\n")

    # Summary by model
    summary = cf.groupby("model").agg(
        avg_coverage=("empirical_coverage", "mean"),
        median_coverage=("empirical_coverage", "median"),
        avg_set_size=("avg_set_size", "mean"),
        median_set_size=("avg_set_size", "median"),
    ).round(4)
    print("=== Conformal coverage by model (target = 0.90) ===")
    print(summary.to_string())

## 6. Computational cost analysis

Aggregate `train_time_s` from existing results CSV. One training time per (task, model, seed), regardless of calibrator.

In [ ]:
# Dedup to one row per (task, model, seed) since train_time_s is shared across calibrators
unique_train = results.drop_duplicates(subset=["task_id", "model", "seed"])

print("=== Training time by model (seconds) ===")
tt = unique_train.groupby("model")["train_time_s"].describe().round(2)
print(tt.to_string())

# Total compute by model
print("\n=== Total training wall-clock by model (hours) ===")
totals = unique_train.groupby("model")["train_time_s"].sum() / 3600
print(totals.round(3).sort_values().to_string())

# Inference scaling check: MC-Dropout has T=30 forward passes
# (We don't time this in the existing CSV, but flag it)
print("\nNote: training time only. Inference cost:")
print(f"  MC-Dropout: ~30× SingleMLP inference (T=30 forward passes)")
print(f"  Ensemble M=5: ~5× SingleMLP inference (no parallelism)")
print(f"  Ensemble M=10: ~10× SingleMLP inference")

# Save table
cost_table = tt[["mean", "std", "50%", "min", "max", "count"]]
cost_table.columns = ["mean_s", "std_s", "median_s", "min_s", "max_s", "n_runs"]
cost_table["total_h"] = totals.round(3)
cost_table.to_csv(WORK_DIR / "computational_cost.csv")
print("\n✓ Saved: computational_cost.csv")

## 7. Imbalance stratification

Stratify 36 datasets into low / medium / high imbalance buckets by max-class / min-class ratio. Re-aggregate per-(model, calibrator) medians within each bucket to check whether calibration recommendations hold across imbalance regimes.

In [ ]:
# Compute imbalance ratio per dataset
imbalance_info = []
for _, row in manifest.iterrows():
    data = load_task(int(row["task_id"]))
    if data is None: continue
    counts = np.bincount(data["y"])
    counts = counts[counts > 0]
    ratio = counts.max() / counts.min()
    imbalance_info.append({"task_id": int(row["task_id"]), "imbalance_ratio": float(ratio)})

imb_df = pd.DataFrame(imbalance_info)
imb_df["imbalance_bucket"] = pd.cut(
    imb_df["imbalance_ratio"],
    bins=[0, 2, 5, 1e9],
    labels=["low (≤2)", "medium (2–5)", "high (>5)"],
)
print("=== Imbalance distribution ===")
print(imb_df.groupby("imbalance_bucket", observed=True).size().to_string())

imb_df.to_csv(WORK_DIR / "imbalance_buckets.csv", index=False)

# Merge into results
results_imb = results.merge(imb_df[["task_id", "imbalance_ratio", "imbalance_bucket"]], on="task_id")

# Two-stage agg by bucket
per_ds_imb = (results_imb.groupby(["imbalance_bucket","task_id","model","calibrator"], observed=True)
              [["cal_nll","cal_ece_mean","cal_brier_score"]].median().reset_index())

# Median across datasets within each bucket
print("\n=== Per-(bucket, model, calibrator) median NLL ===")
bucket_medians = (per_ds_imb.groupby(["imbalance_bucket","model","calibrator"], observed=True)
                  ["cal_nll"].median().unstack("calibrator").round(4))
print(bucket_medians.to_string())
bucket_medians.to_csv(WORK_DIR / "imbalance_stratified_nll.csv")
print("\n✓ Saved: imbalance_stratified_nll.csv")

## 8. Practical-significance reporting

For every statistical test, add columns indicating practical magnitude alongside statistical significance:
- **% relative change** = 100 × (med_a − med_b) / med_b
- **practically_significant** flag: |% change| > 2% for NLL, |abs diff| > 0.005 for ECE

In [ ]:
# Re-compute hypothesis tests with practical-significance columns
# (Mirrors the run_statistical_analysis.ipynb logic but adds practical-significance flags)
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

per_ds = (results.groupby(["task_id","model","calibrator"])
          [["cal_nll","cal_ece_mean","cal_brier_score"]].median().reset_index())

def paired_test(metric, ma, ca, mb, cb):
    a = per_ds[(per_ds.model==ma)&(per_ds.calibrator==ca)].set_index("task_id")[metric]
    b = per_ds[(per_ds.model==mb)&(per_ds.calibrator==cb)].set_index("task_id")[metric]
    a, b = a.align(b, join="inner")
    if len(a) < 5: return None
    diff = a - b
    if (diff == 0).all():
        return {"n":len(a),"med_a":a.median(),"med_b":b.median(),"med_diff":0.0,"p":1.0,"rbc":0.0}
    try:
        stat, p = wilcoxon(a, b, zero_method="wilcox")
    except ValueError:
        return None
    pos = (diff > 0).sum(); neg = (diff < 0).sum(); tot = pos+neg
    rbc = (pos - neg) / max(tot, 1) if tot > 0 else 0.0
    return {"n":len(a),"med_a":float(a.median()),"med_b":float(b.median()),
            "med_diff":float(diff.median()),"p":float(p),"rbc":float(rbc)}

# Just compute H7 (Dirichlet vs MLR, the new finding) with practical sig
H7_models = ["lgbm","xgboost","catboost","single_mlp","mc_dropout",
             "deep_ensemble","deep_ensemble_m3","deep_ensemble_m10"]
H7_rows = []
for m in H7_models:
    for metric in ["cal_nll","cal_ece_mean","cal_brier_score"]:
        r = paired_test(metric, m, "dirichlet", m, "logistic")
        if r:
            # Practical significance
            rel_change = (r["med_a"] - r["med_b"]) / max(abs(r["med_b"]), 1e-9) * 100
            if metric == "cal_nll":
                pract_sig = abs(rel_change) > 2.0
            elif metric == "cal_ece_mean":
                pract_sig = abs(r["med_diff"]) > 0.005
            else:  # brier
                pract_sig = abs(rel_change) > 2.0
            H7_rows.append({"model":m,"metric":metric,
                            "med_dirichlet":r["med_a"],"med_mlr":r["med_b"],
                            "med_diff":r["med_diff"],"rel_change_pct":rel_change,
                            "p":r["p"],"rbc":r["rbc"],
                            "practically_significant":pract_sig})

H7_df = pd.DataFrame(H7_rows)
_, p_adj, _, _ = multipletests(H7_df["p"].values, alpha=0.05, method="holm")
H7_df["p_adj"] = p_adj
H7_df["statistically_significant"] = H7_df["p_adj"] < 0.05

print("=== H7 with practical-significance columns ===")
print(H7_df.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x,float) else str(x)))
H7_df.to_csv(WORK_DIR / "H7_with_practical_significance.csv", index=False)
print("\n✓ Saved: H7_with_practical_significance.csv")

## 9. Composite cost-function score 

Sweep weight triples (w_NLL, w_ECE, w_Brier) summing to 1 in 0.1 increments. For each weight triple, normalize each metric within model family (min-max) and compute composite S. Report which calibrator wins for each weight triple, and the fraction of the weight simplex covered by each calibrator's dominance region.

In [ ]:
# Composite cost score
medians = (per_ds.groupby(["model","calibrator"])
           [["cal_nll","cal_ece_mean","cal_brier_score"]].median().reset_index())

# Min-max normalize within each model
def normalize_within_model(df):
    out = []
    for model, grp in df.groupby("model"):
        g = grp.copy()
        for metric in ["cal_nll","cal_ece_mean","cal_brier_score"]:
            vmin, vmax = g[metric].min(), g[metric].max()
            if vmax > vmin:
                g[f"{metric}_norm"] = (g[metric] - vmin) / (vmax - vmin)
            else:
                g[f"{metric}_norm"] = 0.0
        out.append(g)
    return pd.concat(out, ignore_index=True)

med_norm = normalize_within_model(medians)

# Weight sweep
weights = []
for w1 in np.arange(0, 1.01, 0.1):
    for w2 in np.arange(0, 1.01 - w1, 0.1):
        w3 = 1 - w1 - w2
        if w3 < -1e-9: continue
        w3 = max(w3, 0.0)
        weights.append((round(w1, 1), round(w2, 1), round(w3, 1)))
print(f"Weight triples: {len(weights)}")

# For each weight, find best calibrator per model
dominance = []
for w1, w2, w3 in weights:
    med_norm["S"] = w1*med_norm["cal_nll_norm"] + w2*med_norm["cal_ece_mean_norm"] + w3*med_norm["cal_brier_score_norm"]
    best = med_norm.loc[med_norm.groupby("model")["S"].idxmin()][["model","calibrator","S"]]
    for _, row in best.iterrows():
        dominance.append({"w_nll":w1, "w_ece":w2, "w_brier":w3,
                          "model":row["model"], "winning_calibrator":row["calibrator"]})

dom_df = pd.DataFrame(dominance)

print("\n=== Calibrator dominance region by model ===")
print("(fraction of weight simplex where each calibrator wins)")
for model in sorted(dom_df["model"].unique()):
    sub = dom_df[dom_df.model == model]
    total = len(sub)
    counts = sub["winning_calibrator"].value_counts(normalize=True).round(3)
    print(f"\n{model}:")
    print(counts.to_string())

dom_df.to_csv(WORK_DIR / "composite_score_dominance.csv", index=False)
print("\n✓ Saved: composite_score_dominance.csv")

## 10. Save summary and final outputs

In [ ]:
# Write a summary manifest of what Part 3 produced
import json
from datetime import datetime

summary = {
    "completed_at": datetime.now().isoformat(),
    "files_produced": [],
}

for fname in ["tabpfn_results.csv",
              "ood_results.csv",
              "label_noise_results.csv",
              "conformal_results.csv",
              "computational_cost.csv",
              "imbalance_buckets.csv",
              "imbalance_stratified_nll.csv",
              "H7_with_practical_significance.csv",
              "composite_score_dominance.csv"]:
    p = WORK_DIR / fname
    if p.exists():
        summary["files_produced"].append({"file": fname, "size_bytes": p.stat().st_size})

# Also report archived pre-fix artifact if present
archived = WORK_DIR / "ood_results_pre_rng_fix.csv"
if archived.exists():
    summary["archived_pre_fix"] = {"file": archived.name, "size_bytes": archived.stat().st_size}

# Count saved noise tensors (one per task)
noise_dir = WORK_DIR / "ood_noise"
if noise_dir.exists():
    summary["ood_noise_tensors"] = len(list(noise_dir.glob("task_*.npz")))

print(json.dumps(summary, indent=2))

with open(WORK_DIR / "part3_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(f"\n✓ Summary saved: part3_summary.json")
print(f"\nAll Part 3 outputs are in: {WORK_DIR}")

**Post-run sanity checks (run these once the patched OOD finishes):**

```python
import pandas as pd, numpy as np, pathlib
WORK_DIR = pathlib.Path(os.environ.get("WORK_DIR", "./work")).resolve()

ood = pd.read_csv(WORK_DIR / "ood_results.csv")
print("Rows:", len(ood), "(expected 4320)")
print("Models:", sorted(ood["model"].unique()))
print("Sigmas:", sorted(ood["ood_sigma"].unique()))
print("Any NaN in metric cols:",
      ood[["cal_nll","cal_ece_mean","cal_brier_score"]].isna().any().any())
print("peak_gpu_mem_mb present:", "peak_gpu_mem_mb" in ood.columns,
      "| non-null:", ood["peak_gpu_mem_mb"].notna().sum())

# Spot-check: cached noise tensor for some task should reload byte-identically
tid = int(ood.task_id.iloc[0])
noise = np.load(WORK_DIR / "ood_noise" / f"task_{tid}.npz")
print("Noise file keys:", noise.files,
      "shapes:", {k: noise[k].shape for k in noise.files})
```

